# 🛒 Pakistan's Largest E-Commerce Dataset — EDA + K-Means Clustering

---
## Part 1 — Download the Dataset from Kaggle

**How to get your Kaggle API token (`kaggle.json`):**
1. Log in to [kaggle.com](https://www.kaggle.com)
2. Click your profile picture → **Settings**
3. Scroll to **API** → click **Create New Token**
4. A file named `kaggle.json` will download to your computer

Now run the next cell and upload that `kaggle.json` file.

In [ ]:
# Upload your kaggle.json file
from google.colab import files
uploaded = files.upload()

Move the token to the folder where the Kaggle tool looks for it.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

Make the token private (Kaggle gives a warning if we skip this).

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

Download the dataset from Kaggle.

In [ ]:
!kaggle datasets download -d zusmani/pakistans-largest-ecommerce-dataset

Unzip the downloaded file.

In [ ]:
!unzip -o pakistans-largest-ecommerce-dataset.zip

See which files we got.

In [ ]:
!ls -lh

---
## Part 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

Make the charts look clean and readable.

In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## Part 3 — Load the Data
The CSV file name has spaces in it, so we use `glob` to find it automatically.

In [ ]:
import glob
csv_files = glob.glob("*.csv")
print(csv_files)

In [ ]:
file_path = csv_files[0]
df = pd.read_csv(file_path, low_memory=False)

Keep an untouched copy of the raw data (useful to compare later).

In [ ]:
df_raw = df.copy()

In [ ]:
df.head()

---
## Part 4 — Data Assessment
Before cleaning, we must **understand** the data: size, columns, data types, missing values and duplicates.

### 4.1 How big is the dataset?

In [ ]:
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

### 4.2 Column names

In [ ]:
df.columns.tolist()

### 4.3 Data types and non-null counts

In [ ]:
df.info()

### 4.4 A few random rows (not just the top)

In [ ]:
df.sample(5, random_state=42)

### 4.5 Missing values in each column

In [ ]:
missing_count = df.isnull().sum()
missing_count

Missing values as a **percentage** — easier to judge.

In [ ]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_percent = missing_percent.round(2).sort_values(ascending=False)
missing_percent

Visualize the missing percentage.

In [ ]:
plt.figure(figsize=(12, 5))
missing_percent.plot(kind="bar", color="tomato")
plt.title("Missing Values (%) per Column")
plt.ylabel("Percent Missing")
plt.show()

**🔍 Observation:** Some columns (named `Unnamed: ...`) are almost 100% empty — they are junk columns.
Also, many rows at the bottom of the file are completely empty.

### 4.6 How many rows are completely empty?

In [ ]:
empty_rows = df.isnull().all(axis=1).sum()
print("Completely empty rows:", empty_rows)

### 4.7 Duplicate rows

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

### 4.8 Hidden missing values

This dataset uses the text `\N` to mean *missing*. Pandas does not see it as NaN, so let's count it.

In [ ]:
hidden_missing = (df == "\\N").sum()
hidden_missing[hidden_missing > 0]

### 4.9 Unique values in each column

In [ ]:
df.nunique()

---
## Part 5 — Data Cleaning
We fix problems one by one. Each step has a reason.

### 5.1 Remove the junk `Unnamed` columns

In [ ]:
unnamed_cols = [col for col in df.columns if "Unnamed" in col]
print(unnamed_cols)

In [ ]:
df = df.drop(columns=unnamed_cols)

### 5.2 Remove rows that are completely empty

In [ ]:
df = df.dropna(how="all")
print("Rows now:", df.shape[0])

### 5.3 Clean column names (remove spaces, make lowercase)

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
df.columns.tolist()

### 5.4 Convert hidden missing `\N` into real NaN

In [ ]:
df = df.replace("\\N", np.nan)

### 5.5 Fix the `mv` column

`mv` (money value) is stored as text like `' 1,950 '`. We remove commas and spaces and convert it to a number.

In [ ]:
df["mv"].head()

In [ ]:
df["mv"] = df["mv"].astype(str).str.replace(",", "").str.strip()
df["mv"] = pd.to_numeric(df["mv"], errors="coerce")

In [ ]:
df["mv"].head()

### 5.6 Convert date columns to real dates

In [ ]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["working_date"] = pd.to_datetime(df["working_date"], errors="coerce")

In [ ]:
df[["created_at", "working_date"]].head()

### 5.7 Missing values after the first cleanup

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False)

### 5.8 Plan for filling missing values

| Column | Problem | Approach | Why |
|---|---|---|---|
| `sales_commission_code` | Mostly missing | **Drop column** | Too much missing, not useful |
| `customer_id` | Few missing | **Drop those rows** | Clustering is per customer, we cannot guess an ID |
| `status` | Few missing | Fill with **mode** (most common) | Categorical column |
| `category_name_1` | Some missing | Fill with **"Unknown"** | Keeps rows, honest label |
| `sku` | Few missing | Fill with **"Unknown"** | Product code cannot be guessed |
| `payment_method` | Few missing (if any) | Fill with **mode** | Categorical column |
| `price`, `grand_total`, `discount_amount` | Few missing | Fill with **median** | Median is not affected by outliers |
| `customer_since` | Few missing | Fill with **mode** | Categorical (Year-Month text) |

### 5.9 Drop `sales_commission_code` (too many missing)

In [ ]:
print("Missing %:", round(df["sales_commission_code"].isnull().mean() * 100, 2))

In [ ]:
df = df.drop(columns=["sales_commission_code"])

### 5.10 Drop rows with no `customer_id`

In [ ]:
print("Rows without customer_id:", df["customer_id"].isnull().sum())

In [ ]:
df = df.dropna(subset=["customer_id"])

### 5.11 Fill `status` with the most common value (mode)

In [ ]:
status_mode = df["status"].mode()[0]
print("Most common status:", status_mode)

In [ ]:
df["status"] = df["status"].fillna(status_mode)

### 5.12 Fill `category_name_1` and `sku` with 'Unknown'

In [ ]:
df["category_name_1"] = df["category_name_1"].fillna("Unknown")

In [ ]:
df["sku"] = df["sku"].fillna("Unknown")

### 5.13 Fill `payment_method` with mode

In [ ]:
df["payment_method"] = df["payment_method"].fillna(df["payment_method"].mode()[0])

### 5.14 Fill numeric columns with median

In [ ]:
numeric_cols = ["price", "qty_ordered", "grand_total", "discount_amount", "mv"]

In [ ]:
for col in numeric_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

### 5.15 Fill `customer_since` with mode

In [ ]:
df["customer_since"] = df["customer_since"].fillna(df["customer_since"].mode()[0])

### 5.16 Drop any row where the order date could not be read

In [ ]:
print("Rows without a valid date:", df["created_at"].isnull().sum())

In [ ]:
df = df.dropna(subset=["created_at"])

### 5.17 Remove duplicate rows

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

### 5.18 Fix data types

In [ ]:
df["customer_id"] = df["customer_id"].astype(int)
df["qty_ordered"] = df["qty_ordered"].astype(int)

### 5.19 Remove impossible values (negative price or quantity)

In [ ]:
print("Negative price rows:", (df["price"] < 0).sum())
print("Zero/negative qty rows:", (df["qty_ordered"] <= 0).sum())

In [ ]:
df = df[df["price"] >= 0]
df = df[df["qty_ordered"] > 0]

### 5.20 Final check — no missing values should remain

In [ ]:
df.isnull().sum()

In [ ]:
print("Rows before cleaning:", df_raw.shape[0])
print("Rows after cleaning :", df.shape[0])

---
## Part 6 — Create Helper Columns for Analysis

Extract year, month and weekday from the order date.

In [ ]:
df["order_year"] = df["created_at"].dt.year
df["order_month"] = df["created_at"].dt.month
df["order_weekday"] = df["created_at"].dt.day_name()
df["year_month"] = df["created_at"].dt.to_period("M").astype(str)

Look at all order statuses.

In [ ]:
df["status"].value_counts()

There are many statuses. Let's group them into 4 simple groups.

In [ ]:
def group_status(s):
    if s in ["complete", "received", "closed", "paid", "cod", "exchange"]:
        return "Completed"
    elif s in ["canceled", "fraud"]:
        return "Canceled"
    elif s in ["order_refunded", "refund"]:
        return "Refunded"
    else:
        return "Other"

In [ ]:
df["status_group"] = df["status"].apply(group_status)
df["status_group"].value_counts()

---
## Part 7 — Exploratory Data Analysis (EDA)
We look at the data from many angles: **statistics, orders, categories, payments, time, money and customers.**

### 7.1 Statistical summary of numeric columns

In [ ]:
df[numeric_cols].describe().round(2)

### 7.2 Summary of text (categorical) columns

In [ ]:
df[["status", "category_name_1", "payment_method", "bi_status"]].describe()

### 7.3 Order Status

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, y="status", order=df["status"].value_counts().index, palette="viridis")
plt.title("Number of Orders by Status")
plt.xlabel("Number of Orders")
plt.show()

In [ ]:
df["status_group"].value_counts().plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6))
plt.title("Order Status Groups")
plt.ylabel("")
plt.show()

**🔍 Insight:** A large share of orders get canceled — a big business problem worth studying.

### 7.4 Product Categories

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y="category_name_1", order=df["category_name_1"].value_counts().index, palette="mako")
plt.title("Number of Orders by Category")
plt.xlabel("Number of Orders")
plt.show()

Which category brings the most **revenue**?

In [ ]:
category_revenue = df.groupby("category_name_1")["grand_total"].sum().sort_values(ascending=False)
category_revenue

In [ ]:
plt.figure(figsize=(12, 6))
(category_revenue / 1_000_000).plot(kind="barh", color="teal")
plt.title("Total Revenue by Category (Million PKR)")
plt.xlabel("Revenue (Million PKR)")
plt.gca().invert_yaxis()
plt.show()

Average item price per category.

In [ ]:
avg_price = df.groupby("category_name_1")["price"].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
avg_price.plot(kind="barh", color="orange")
plt.title("Average Item Price by Category (PKR)")
plt.gca().invert_yaxis()
plt.show()

Cancel rate in each category.

In [ ]:
cancel_rate = df.groupby("category_name_1")["status_group"].apply(lambda x: (x == "Canceled").mean() * 100)
cancel_rate = cancel_rate.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(12, 6))
cancel_rate.plot(kind="barh", color="crimson")
plt.title("Cancel Rate (%) by Category")
plt.xlabel("Percent Canceled")
plt.gca().invert_yaxis()
plt.show()

### 7.5 Payment Methods

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, y="payment_method", order=df["payment_method"].value_counts().index, palette="rocket")
plt.title("Orders by Payment Method")
plt.show()

Which payment methods get canceled most? (row percentages)

In [ ]:
pay_status = pd.crosstab(df["payment_method"], df["status_group"], normalize="index") * 100
pay_status.round(1)

In [ ]:
pay_status.plot(kind="barh", stacked=True, figsize=(12, 6), colormap="Set2")
plt.title("Order Outcome by Payment Method (%)")
plt.xlabel("Percent")
plt.legend(bbox_to_anchor=(1.02, 1))
plt.show()

### 7.6 Sales Over Time

In [ ]:
monthly_orders = df.groupby("year_month")["increment_id"].nunique()

In [ ]:
plt.figure(figsize=(14, 5))
monthly_orders.plot(marker="o")
plt.title("Number of Orders per Month")
plt.ylabel("Orders")
plt.xticks(rotation=90)
plt.show()

In [ ]:
monthly_revenue = df.groupby("year_month")["grand_total"].sum() / 1_000_000

In [ ]:
plt.figure(figsize=(14, 5))
monthly_revenue.plot(marker="o", color="green")
plt.title("Revenue per Month (Million PKR)")
plt.ylabel("Revenue (Million PKR)")
plt.xticks(rotation=90)
plt.show()

**🔍 Insight:** Look for big spikes — usually sale events like **11.11** and **Black Friday (November)**.

Orders by year.

In [ ]:
sns.countplot(data=df, x="order_year", palette="Blues")
plt.title("Order Lines per Year")
plt.show()

Orders by day of the week.

In [ ]:
week_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sns.countplot(data=df, x="order_weekday", order=week_order, palette="coolwarm")
plt.title("Orders by Day of Week")
plt.xticks(rotation=45)
plt.show()

Heatmap: orders by month and year.

In [ ]:
month_year = pd.crosstab(df["order_month"], df["order_year"])

In [ ]:
plt.figure(figsize=(8, 7))
sns.heatmap(month_year, annot=True, fmt="d", cmap="YlOrRd")
plt.title("Orders by Month and Year")
plt.show()

### 7.7 Money Columns — Distributions and Outliers

In [ ]:
sns.histplot(df["price"], bins=50, color="purple")
plt.title("Price Distribution (raw)")
plt.show()

The data is very **skewed** (a few very expensive items). A log scale shows it better.

In [ ]:
sns.histplot(np.log1p(df["price"]), bins=50, color="purple")
plt.title("Price Distribution (log scale)")
plt.xlabel("log(1 + price)")
plt.show()

In [ ]:
sns.histplot(np.log1p(df["grand_total"]), bins=50, color="green")
plt.title("Grand Total Distribution (log scale)")
plt.show()

In [ ]:
sns.boxplot(x=df["grand_total"])
plt.title("Grand Total — Boxplot (dots are outliers)")
plt.show()

How many orders got a discount?

In [ ]:
has_discount = (df["discount_amount"] > 0).mean() * 100
print(f"Orders with discount: {has_discount:.1f}%")

In [ ]:
sns.histplot(df.loc[df["discount_amount"] > 0, "discount_amount"].clip(upper=5000), bins=50, color="gold")
plt.title("Discount Amount (only discounted orders, capped at 5000)")
plt.show()

Quantity per order line.

In [ ]:
df["qty_ordered"].value_counts().head(10).plot(kind="bar", color="slateblue")
plt.title("Quantity Ordered (Top 10 values)")
plt.show()

### 7.8 Correlation Between Numeric Columns

In [ ]:
corr = df[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

### 7.9 Customers

In [ ]:
print("Unique customers:", df["customer_id"].nunique())
print("Unique orders   :", df["increment_id"].nunique())

In [ ]:
orders_per_customer = df.groupby("customer_id")["increment_id"].nunique()
orders_per_customer.describe()

In [ ]:
orders_per_customer.clip(upper=20).plot(kind="hist", bins=20, color="skyblue")
plt.title("Orders per Customer (capped at 20)")
plt.xlabel("Number of Orders")
plt.show()

**🔍 Insight:** Most customers buy only once or twice — a few loyal customers buy a lot. That is exactly what clustering will help us find.

---
## Part 8 — Build Customer Features for Clustering (RFM)

K-Means needs **one row per customer**. We use the famous **RFM** idea:

- **Recency** → how many days since the customer's last order (smaller = better)
- **Frequency** → how many orders the customer made
- **Monetary** → how much money the customer spent

We also add **average order value**, **total quantity** and **total discount**.

> We use only **completed** orders, because canceled orders are not real purchases.

In [ ]:
completed = df[df["status_group"] == "Completed"]
print("Completed order lines:", completed.shape[0])

Reference date = one day after the last order in the data.

In [ ]:
reference_date = completed["created_at"].max() + pd.Timedelta(days=1)
print(reference_date)

In [ ]:
customers = completed.groupby("customer_id").agg(
    last_order=("created_at", "max"),
    frequency=("increment_id", "nunique"),
    monetary=("grand_total", "sum"),
    total_qty=("qty_ordered", "sum"),
    total_discount=("discount_amount", "sum")
)

In [ ]:
customers["recency"] = (reference_date - customers["last_order"]).dt.days

In [ ]:
customers["avg_order_value"] = customers["monetary"] / customers["frequency"]

In [ ]:
customers = customers.drop(columns=["last_order"])
customers.head()

In [ ]:
print("Number of customers:", customers.shape[0])

In [ ]:
customers.describe().round(2)

Check: any missing values in the customer table?

In [ ]:
customers.isnull().sum()

Remove customers who spent 0 (they would confuse the model).

In [ ]:
customers = customers[customers["monetary"] > 0]

---
## Part 9 — Make the Data Reliable for K-Means

K-Means uses **distance**. So it is sensitive to:
1. **Outliers** — one huge customer can pull a whole cluster
2. **Skewed data** — most values small, a few very large
3. **Different scales** — money in thousands vs. frequency in single digits

We fix each problem one by one.

In [ ]:
features = ["recency", "frequency", "monetary", "avg_order_value", "total_qty", "total_discount"]

### 9.1 Look at the distributions before fixing

In [ ]:
customers[features].hist(bins=40, figsize=(14, 8), color="steelblue")
plt.suptitle("Feature Distributions — BEFORE")
plt.show()

### 9.2 Cap outliers (IQR method)

Values above `Q3 + 1.5 × IQR` are pulled down to that limit. We **cap** instead of delete, so we keep every customer.

In [ ]:
X = customers[features].copy()

In [ ]:
for col in features:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    X[col] = X[col].clip(lower=lower, upper=upper)

> Note: `frequency` and `total_discount` can become constant after IQR capping (most customers have 1 order / 0 discount). We check this next.

In [ ]:
X.nunique()

If a column has only 1 unique value now, it carries no information. To keep that information we use the **99th percentile** cap for those columns instead.

In [ ]:
for col in features:
    if X[col].nunique() <= 1:
        upper = customers[col].quantile(0.99)
        X[col] = customers[col].clip(upper=upper)
        print("Used 99th percentile cap for:", col)

### 9.3 Log transform to reduce skew

In [ ]:
X_log = np.log1p(X)

In [ ]:
X_log.hist(bins=40, figsize=(14, 8), color="seagreen")
plt.suptitle("Feature Distributions — AFTER capping + log")
plt.show()

### 9.4 Scale features (mean = 0, std = 1)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

In [ ]:
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.describe().round(2)

Check correlation between the final features.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X_scaled_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation of Clustering Features")
plt.show()

---
## Part 10 — K-Means Clustering

### 10.1 Elbow Method
We train K-Means for K = 2 to 10 and record the **inertia** (how tight the clusters are).
The best K is where the line bends like an **elbow**.

In [ ]:
inertia = []
k_values = range(2, 11)

In [ ]:
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)
    print("K =", k, "done")

In [ ]:
plt.plot(k_values, inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.show()

### 10.2 Silhouette Score
Score between -1 and 1. **Higher = better separated clusters.**
Silhouette is slow on big data, so we calculate it on a random sample of 10,000 customers.

In [ ]:
sample_size = min(10000, len(X_scaled))
np.random.seed(42)
sample_index = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_index]

In [ ]:
sil_scores = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_sample)
    score = silhouette_score(X_sample, labels)
    sil_scores.append(score)
    print("K =", k, " Silhouette =", round(score, 3))

In [ ]:
plt.plot(k_values, sil_scores, marker="o", color="darkorange")
plt.title("Silhouette Score for each K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.show()

### 10.3 Choose K
Look at both charts. Pick a K where the elbow bends **and** silhouette is good.
For customer segmentation, **K = 4** is usually a good, easy-to-explain choice.
👉 Change the number below if your charts suggest a different K.

In [ ]:
best_k = 4

### 10.4 Train the final model

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans.fit(X_scaled)

In [ ]:
customers["cluster"] = kmeans.labels_

In [ ]:
customers["cluster"].value_counts().sort_index()

In [ ]:
sns.countplot(data=customers, x="cluster", palette="Set2")
plt.title("Number of Customers in each Cluster")
plt.show()

---
## Part 11 — Understand the Clusters

### 11.1 Average values per cluster (original units)

In [ ]:
cluster_profile = customers.groupby("cluster")[features].mean().round(1)
cluster_profile

Add the cluster size.

In [ ]:
cluster_profile["customers"] = customers["cluster"].value_counts().sort_index()
cluster_profile

### 11.2 Heatmap of the cluster profile (scaled so colors are comparable)

In [ ]:
profile_scaled = X_scaled_df.copy()
profile_scaled["cluster"] = kmeans.labels_
profile_scaled = profile_scaled.groupby("cluster").mean()

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(profile_scaled, annot=True, cmap="RdYlGn", fmt=".2f")
plt.title("Cluster Profile (scaled) — green = high, red = low")
plt.show()

### 11.3 Boxplots of each feature by cluster

In [ ]:
for col in ["recency", "frequency", "monetary", "avg_order_value"]:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=customers, x="cluster", y=col, palette="Set2", showfliers=False)
    plt.title(f"{col} by Cluster")
    plt.show()

### 11.4 Visualize clusters in 2D with PCA

PCA squeezes our 6 features into 2, so we can draw the clusters on a flat chart.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
print("Variance explained by 2 components:", round(pca.explained_variance_ratio_.sum() * 100, 1), "%")

In [ ]:
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["cluster"] = kmeans.labels_

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df.sample(20000, random_state=42), x="PC1", y="PC2",
                hue="cluster", palette="Set2", s=10, alpha=0.6)
plt.title("Customer Clusters (PCA view)")
plt.show()

### 11.5 Recency vs Monetary scatter

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=customers.sample(20000, random_state=42), x="recency", y="monetary",
                hue="cluster", palette="Set2", s=10, alpha=0.6)
plt.yscale("log")
plt.title("Recency vs Monetary by Cluster")
plt.show()

### 11.6 Give each cluster a business name
Look at the profile table above and name each cluster. Typical segments:

| Pattern | Name |
|---|---|
| Low recency, high frequency, high monetary | 🏆 **Champions / Loyal** |
| High avg order value, few orders | 💎 **Big Spenders** |
| Low recency, 1 order, low spend | 🌱 **New Customers** |
| High recency, low frequency, low spend | 😴 **Lost / Inactive** |

👉 Edit the dictionary below to match **your** results (cluster numbers can change between runs).

In [ ]:
cluster_names = {
    0: "Cluster 0",
    1: "Cluster 1",
    2: "Cluster 2",
    3: "Cluster 3"
}

In [ ]:
customers["segment"] = customers["cluster"].map(cluster_names)
customers["segment"].value_counts()

Revenue share of each segment.

In [ ]:
segment_revenue = customers.groupby("segment")["monetary"].sum()
segment_revenue.plot(kind="pie", autopct="%1.1f%%", figsize=(7, 7))
plt.title("Revenue Share by Segment")
plt.ylabel("")
plt.show()

---
## Part 12 — Save the Results

In [ ]:
customers.to_csv("customer_segments.csv")

In [ ]:
files.download("customer_segments.csv")

###  Practice tasks for students
- Try K = 3 and K = 5. Which one gives easier-to-explain groups?
- Remove `total_discount` from the features. Do the clusters change?
- Find the most popular category inside each cluster.